# **Result**

In [1]:
# ============================================================
# COMPARISON NOTEBOOK - Cell 1: Load Results
# ============================================================
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for compatibility
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available
              else 'ggplot')

# Define colors for each framework
COLORS = {
    'PyTorch': '#EE4C2C',
    'TensorFlow': '#FF6F00',
    'JAX': '#A855F7',
}

# Load results
data = {}
files = {
    'PyTorch': 'benchmark_pytorch_results.json',
    'TensorFlow': 'benchmark_tensorflow_results.json',
    'JAX': 'benchmark_jax_results.json',
}

for name, filepath in files.items():
    try:
        with open(filepath, 'r') as f:
            data[name] = json.load(f)
        print(f"✅ Loaded {name}: {filepath}")
        print(f"   Version: {data[name].get('version', 'N/A')}")
        print(f"   Device: {data[name].get('device', 'N/A')}")
        print(f"   Timestamp: {data[name].get('timestamp', 'N/A')}")
    except FileNotFoundError:
        print(f"❌ {name}: {filepath} not found!")
    except json.JSONDecodeError:
        print(f"❌ {name}: {filepath} is not valid JSON!")

frameworks = list(data.keys())
print(f"\nFrameworks loaded: {frameworks}")

def safe_get(d, *keys, default=None):
    """Safely navigate nested dict"""
    for key in keys:
        if isinstance(d, dict):
            d = d.get(key, default)
        else:
            return default
        if d is None:
            return default
    return d

✅ Loaded PyTorch: benchmark_pytorch_results.json
   Version: 2.10.0+rocm7.1
   Device: AMD Radeon RX 6800S
   Timestamp: 2026-02-17T11:32:06.496167
✅ Loaded TensorFlow: benchmark_tensorflow_results.json
   Version: 2.20.0-dev0+selfbuilt
   Device: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
   Timestamp: 2026-02-17T12:18:06.365902
✅ Loaded JAX: benchmark_jax_results.json
   Version: 0.7.1
   Device: rocm:0
   Timestamp: 2026-02-17T14:01:58.752199

Frameworks loaded: ['PyTorch', 'TensorFlow', 'JAX']


In [ ]:
# ============================================================
# COMPARISON - Cell 2: Matrix Multiplication
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Matrix Multiplication Performance Comparison', fontsize=18, fontweight='bold')

sizes = ['256', '512', '1024', '2048', '4096', '8192']
size_labels = [f'{s}' for s in sizes]

# --- FP32 Time ---
ax = axes[0]
x_pos = np.arange(len(sizes))
bar_width = 0.25
offsets = np.array([-1, 0, 1]) * bar_width

for i, fw in enumerate(frameworks):
    times = []
    for s in sizes:
        val = safe_get(data[fw], 'benchmarks', 'matmul', 'float32', s, 'mean_ms', default=0)
        times.append(val if val else 0)
    mask = [t > 0 for t in times]
    positions = [x_pos[j] + offsets[i] for j in range(len(sizes)) if mask[j]]
    values = [t for t in times if t > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Matrix Size (NxN)', fontsize=12)
ax.set_ylabel('Time (ms) — lower is better', fontsize=12)
ax.set_title('FP32 Matrix Multiply Time', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(size_labels)
ax.legend()
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# --- FP32 TFLOPS ---
ax = axes[1]
for i, fw in enumerate(frameworks):
    tflops = []
    valid_sizes = []
    for j, s in enumerate(sizes):
        val = safe_get(data[fw], 'benchmarks', 'matmul', 'float32', s, 'tflops', default=0)
        if val and val > 0:
            tflops.append(val)
            valid_sizes.append(x_pos[j] + offsets[i])
    if tflops:
        ax.bar(valid_sizes, tflops, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Matrix Size (NxN)', fontsize=12)
ax.set_ylabel('TFLOPS — higher is better', fontsize=12)
ax.set_title('FP32 Matrix Multiply Throughput', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(size_labels)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# --- FP16 TFLOPS ---
ax = axes[2]
for i, fw in enumerate(frameworks):
    tflops = []
    valid_sizes = []
    for j, s in enumerate(sizes):
        val = safe_get(data[fw], 'benchmarks', 'matmul', 'float16', s, 'tflops', default=0)
        if val and val > 0:
            tflops.append(val)
            valid_sizes.append(x_pos[j] + offsets[i])
    if tflops:
        ax.bar(valid_sizes, tflops, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Matrix Size (NxN)', fontsize=12)
ax.set_ylabel('TFLOPS — higher is better', fontsize=12)
ax.set_title('FP16 Matrix Multiply Throughput', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(size_labels)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_matmul.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: comparison_matmul.png")

✅ Saved: comparison_matmul.png


In [3]:
# ============================================================
# COMPARISON - Cell 3: Element-wise Operations
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Element-wise & Activation Function Performance (50M elements)',
             fontsize=16, fontweight='bold')

target_size = '50M'
ops_groups = {
    'Arithmetic': ['add', 'mul', 'exp', 'sin'],
    'Activations': ['sigmoid', 'tanh', 'relu', 'gelu', 'silu'],
}

# Time comparison
for idx, (group_name, ops) in enumerate(ops_groups.items()):
    ax = axes[0][idx]
    x_pos = np.arange(len(ops))
    bar_width = 0.25
    offsets = np.array([-1, 0, 1]) * bar_width

    for i, fw in enumerate(frameworks):
        times = []
        for op in ops:
            val = safe_get(data[fw], 'benchmarks', 'elementwise', target_size, op, 'mean_ms', default=0)
            times.append(val if val else 0)
        mask = [t > 0 for t in times]
        positions = [x_pos[j] + offsets[i] for j in range(len(ops)) if mask[j]]
        values = [t for t in times if t > 0]
        if values:
            ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

    ax.set_xlabel('Operation', fontsize=11)
    ax.set_ylabel('Time (ms) — lower is better', fontsize=11)
    ax.set_title(f'{group_name} Operations', fontsize=13)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(ops, rotation=30)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

# Reduction operations
reduction_ops = ['sum', 'mean', 'max', 'std', 'argmax', 'sort']
ax = axes[1][0]
x_pos = np.arange(len(reduction_ops))
bar_width = 0.25
offsets = np.array([-1, 0, 1]) * bar_width

for i, fw in enumerate(frameworks):
    times = []
    for op in reduction_ops:
        val = safe_get(data[fw], 'benchmarks', 'reductions', target_size, op, 'mean_ms', default=0)
        times.append(val if val else 0)
    mask = [t > 0 for t in times]
    positions = [x_pos[j] + offsets[i] for j in range(len(reduction_ops)) if mask[j]]
    values = [t for t in times if t > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Operation', fontsize=11)
ax.set_ylabel('Time (ms) — lower is better', fontsize=11)
ax.set_title(f'Reduction Operations ({target_size} elements)', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(reduction_ops, rotation=30)
ax.legend(fontsize=9)
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# Bandwidth comparison for element-wise
ax = axes[1][1]
all_ops = ['add', 'mul', 'exp', 'relu', 'gelu']
x_pos = np.arange(len(all_ops))

for i, fw in enumerate(frameworks):
    bw = []
    for op in all_ops:
        val = safe_get(data[fw], 'benchmarks', 'elementwise', target_size, op, 'bandwidth_gbs', default=0)
        bw.append(val if val else 0)
    mask = [b > 0 for b in bw]
    positions = [x_pos[j] + offsets[i] for j in range(len(all_ops)) if mask[j]]
    values = [b for b in bw if b > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Operation', fontsize=11)
ax.set_ylabel('Effective Bandwidth (GB/s) — higher is better', fontsize=11)
ax.set_title(f'Memory Bandwidth Utilization ({target_size} elements)', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(all_ops, rotation=30)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_elementwise.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: comparison_elementwise.png")

✅ Saved: comparison_elementwise.png


In [4]:
# ============================================================
# COMPARISON - Cell 4: Convolution & Layer Operations
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('Convolution & Layer Operations Performance', fontsize=16, fontweight='bold')

# Conv2D Forward
conv_labels = ["ResNet-stem", "ResNet-block1", "ResNet-block2", "ResNet-block3",
               "ResNet-block4", "CIFAR-style", "VGG-style"]

ax = axes[0][0]
x_pos = np.arange(len(conv_labels))
bar_width = 0.25
offsets = np.array([-1, 0, 1]) * bar_width

for i, fw in enumerate(frameworks):
    times = []
    for cl in conv_labels:
        val = safe_get(data[fw], 'benchmarks', 'conv2d', cl, 'forward', 'mean_ms', default=0)
        times.append(val if val else 0)
    mask = [t > 0 for t in times]
    positions = [x_pos[j] + offsets[i] for j in range(len(conv_labels)) if mask[j]]
    values = [t for t in times if t > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_ylabel('Time (ms) — lower is better', fontsize=11)
ax.set_title('Conv2D Forward Pass', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(conv_labels, rotation=35, ha='right', fontsize=9)
ax.legend(fontsize=9)
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# Conv2D Forward+Backward
ax = axes[0][1]
for i, fw in enumerate(frameworks):
    times = []
    for cl in conv_labels:
        val = safe_get(data[fw], 'benchmarks', 'conv2d', cl, 'fwd_bwd', 'mean_ms', default=0)
        times.append(val if val else 0)
    mask = [t > 0 for t in times]
    positions = [x_pos[j] + offsets[i] for j in range(len(conv_labels)) if mask[j]]
    values = [t for t in times if t > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_ylabel('Time (ms) — lower is better', fontsize=11)
ax.set_title('Conv2D Forward + Backward', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(conv_labels, rotation=35, ha='right', fontsize=9)
ax.legend(fontsize=9)
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# Attention layers
attn_labels = ["BERT-attn", "BERT-long-attn", "Large-attn", "LLM-attn", "Long-seq-attn"]
ax = axes[1][0]
x_pos = np.arange(len(attn_labels))

for i, fw in enumerate(frameworks):
    times = []
    for al in attn_labels:
        val = safe_get(data[fw], 'benchmarks', 'layers', 'attention', al, 'forward', 'mean_ms', default=0)
        times.append(val if val else 0)
    mask = [t > 0 for t in times]
    positions = [x_pos[j] + offsets[i] for j in range(len(attn_labels)) if mask[j]]
    values = [t for t in times if t > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_ylabel('Time (ms) — lower is better', fontsize=11)
ax.set_title('Multi-Head Attention Forward', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(attn_labels, rotation=30, ha='right', fontsize=9)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Linear + LayerNorm + BatchNorm
misc_layers = {
    'Linear': ('layers', 'linear', 'BERT-hidden', 'forward'),
    'Linear-FFN': ('layers', 'linear', 'BERT-FFN-up', 'forward'),
    'BatchNorm': ('layers', 'batchnorm', 'BN-early'),
    'LayerNorm': ('layers', 'layernorm', 'BERT-LN'),
    'Softmax': ('layers', 'softmax', '128x128x768'),
}

ax = axes[1][1]
layer_names = list(misc_layers.keys())
x_pos = np.arange(len(layer_names))

for i, fw in enumerate(frameworks):
    times = []
    for lname, path in misc_layers.items():
        if len(path) == 4:
            val = safe_get(data[fw], 'benchmarks', *path, 'mean_ms', default=0)
        else:
            val = safe_get(data[fw], 'benchmarks', *path, 'mean_ms', default=0)
        times.append(val if val else 0)
    mask = [t > 0 for t in times]
    positions = [x_pos[j] + offsets[i] for j in range(len(layer_names)) if mask[j]]
    values = [t for t in times if t > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_ylabel('Time (ms) — lower is better', fontsize=11)
ax.set_title('Misc Layer Operations Forward', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(layer_names, rotation=30, ha='right')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_layers.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: comparison_layers.png")

✅ Saved: comparison_layers.png


In [5]:
# ============================================================
# COMPARISON - Cell 5: Model Training Performance
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(24, 14))
fig.suptitle('End-to-End Model Training Performance', fontsize=18, fontweight='bold')

bar_width = 0.25
offsets = np.array([-1, 0, 1]) * bar_width

# --- CNN Inference Throughput ---
ax = axes[0][0]
batch_labels = ['batch_16', 'batch_32', 'batch_64']
display_labels = ['BS=16', 'BS=32', 'BS=64']
x_pos = np.arange(len(batch_labels))

for i, fw in enumerate(frameworks):
    throughputs = []
    for bl in batch_labels:
        val = safe_get(data[fw], 'benchmarks', 'cnn_training', bl, 'inference',
                      'throughput_imgs_per_sec', default=0)
        throughputs.append(val if val else 0)
    mask = [t > 0 for t in throughputs]
    positions = [x_pos[j] + offsets[i] for j in range(len(batch_labels)) if mask[j]]
    values = [t for t in throughputs if t > 0]
    if values:
        bars = ax.bar(positions, values, bar_width * 0.9, label=fw,
                     color=COLORS.get(fw, 'gray'), alpha=0.85)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                   f'{val:.0f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Batch Size', fontsize=11)
ax.set_ylabel('Images/sec — higher is better', fontsize=11)
ax.set_title('ResNet-18 Inference Throughput', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(display_labels)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# --- CNN Training Throughput ---
ax = axes[0][1]
for i, fw in enumerate(frameworks):
    throughputs = []
    for bl in batch_labels:
        val = safe_get(data[fw], 'benchmarks', 'cnn_training', bl, 'training',
                      'throughput_imgs_per_sec', default=0)
        throughputs.append(val if val else 0)
    mask = [t > 0 for t in throughputs]
    positions = [x_pos[j] + offsets[i] for j in range(len(batch_labels)) if mask[j]]
    values = [t for t in throughputs if t > 0]
    if values:
        bars = ax.bar(positions, values, bar_width * 0.9, label=fw,
                     color=COLORS.get(fw, 'gray'), alpha=0.85)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                   f'{val:.0f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Batch Size', fontsize=11)
ax.set_ylabel('Images/sec — higher is better', fontsize=11)
ax.set_title('ResNet-18 Training Throughput', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(display_labels)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# --- CNN Training Time per Step ---
ax = axes[0][2]
for i, fw in enumerate(frameworks):
    times = []
    for bl in batch_labels:
        val = safe_get(data[fw], 'benchmarks', 'cnn_training', bl, 'training', 'mean_ms', default=0)
        times.append(val if val else 0)
    mask = [t > 0 for t in times]
    positions = [x_pos[j] + offsets[i] for j in range(len(batch_labels)) if mask[j]]
    values = [t for t in times if t > 0]
    if values:
        bars = ax.bar(positions, values, bar_width * 0.9, label=fw,
                     color=COLORS.get(fw, 'gray'), alpha=0.85)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                   f'{val:.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Batch Size', fontsize=11)
ax.set_ylabel('Time/step (ms) — lower is better', fontsize=11)
ax.set_title('ResNet-18 Training Step Time', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(display_labels)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# --- Transformer Inference ---
trans_labels = ["Small-Transformer", "BERT-base-like", "BERT-long-seq", "BERT-large-like"]
trans_short = ["Small", "BERT-base", "BERT-long", "BERT-large"]
x_pos_t = np.arange(len(trans_labels))

ax = axes[1][0]
for i, fw in enumerate(frameworks):
    tokens = []
    for tl in trans_labels:
        val = safe_get(data[fw], 'benchmarks', 'transformer_training', tl, 'inference',
                      'tokens_per_sec', default=0)
        tokens.append(val if val else 0)
    mask = [t > 0 for t in tokens]
    positions = [x_pos_t[j] + offsets[i] for j in range(len(trans_labels)) if mask[j]]
    values = [t for t in tokens if t > 0]
    if values:
        bars = ax.bar(positions, values, bar_width * 0.9, label=fw,
                     color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Model Configuration', fontsize=11)
ax.set_ylabel('Tokens/sec — higher is better', fontsize=11)
ax.set_title('Transformer Inference Throughput', fontsize=13)
ax.set_xticks(x_pos_t)
ax.set_xticklabels(trans_short, rotation=20)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# --- Transformer Training ---
ax = axes[1][1]
for i, fw in enumerate(frameworks):
    tokens = []
    for tl in trans_labels:
        val = safe_get(data[fw], 'benchmarks', 'transformer_training', tl, 'training',
                      'tokens_per_sec', default=0)
        tokens.append(val if val else 0)
    mask = [t > 0 for t in tokens]
    positions = [x_pos_t[j] + offsets[i] for j in range(len(trans_labels)) if mask[j]]
    values = [t for t in tokens if t > 0]
    if values:
        bars = ax.bar(positions, values, bar_width * 0.9, label=fw,
                     color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Model Configuration', fontsize=11)
ax.set_ylabel('Tokens/sec — higher is better', fontsize=11)
ax.set_title('Transformer Training Throughput', fontsize=13)
ax.set_xticks(x_pos_t)
ax.set_xticklabels(trans_short, rotation=20)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# --- Mixed Precision Comparison ---
ax = axes[1][2]
metrics = ['FP32 Train', 'AMP Train']
x_pos_m = np.arange(len(metrics))

for i, fw in enumerate(frameworks):
    values = []
    # FP32 batch=32
    val_fp32 = safe_get(data[fw], 'benchmarks', 'cnn_training', 'batch_32', 'training',
                       'throughput_imgs_per_sec', default=0)
    values.append(val_fp32 if val_fp32 else 0)
    # AMP batch=32
    val_amp = safe_get(data[fw], 'benchmarks', 'cnn_training', 'amp_batch_32', 'training',
                      'throughput_imgs_per_sec', default=0)
    values.append(val_amp if val_amp else 0)

    mask = [v > 0 for v in values]
    positions = [x_pos_m[j] + offsets[i] for j in range(len(metrics)) if mask[j]]
    valid = [v for v in values if v > 0]
    if valid:
        bars = ax.bar(positions, valid, bar_width * 0.9, label=fw,
                     color=COLORS.get(fw, 'gray'), alpha=0.85)
        for bar, val in zip(bars, valid):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                   f'{val:.0f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Precision', fontsize=11)
ax.set_ylabel('Images/sec — higher is better', fontsize=11)
ax.set_title('FP32 vs Mixed Precision (ResNet, BS=32)', fontsize=13)
ax.set_xticks(x_pos_m)
ax.set_xticklabels(metrics)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_training.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: comparison_training.png")

✅ Saved: comparison_training.png


In [6]:
# ============================================================
# COMPARISON - Cell 6: Memory Transfer & Final Summary
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Memory Transfer & Data Movement', fontsize=16, fontweight='bold')

bar_width = 0.25
offsets = np.array([-1, 0, 1]) * bar_width

# H2D Bandwidth
transfer_sizes = ['1MB', '10MB', '50MB', '100MB', '500MB', '1000MB']
ax = axes[0]
x_pos = np.arange(len(transfer_sizes))

for i, fw in enumerate(frameworks):
    bw = []
    for ts in transfer_sizes:
        val = safe_get(data[fw], 'benchmarks', 'memory', 'h2d', ts, 'bandwidth_gbs', default=0)
        bw.append(val if val else 0)
    mask = [b > 0 for b in bw]
    positions = [x_pos[j] + offsets[i] for j in range(len(transfer_sizes)) if mask[j]]
    values = [b for b in bw if b > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Transfer Size', fontsize=11)
ax.set_ylabel('Bandwidth (GB/s) — higher is better', fontsize=11)
ax.set_title('Host → Device Transfer', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(transfer_sizes, rotation=30)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# D2H Bandwidth
ax = axes[1]
for i, fw in enumerate(frameworks):
    bw = []
    for ts in transfer_sizes:
        val = safe_get(data[fw], 'benchmarks', 'memory', 'd2h', ts, 'bandwidth_gbs', default=0)
        bw.append(val if val else 0)
    mask = [b > 0 for b in bw]
    positions = [x_pos[j] + offsets[i] for j in range(len(transfer_sizes)) if mask[j]]
    values = [b for b in bw if b > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Transfer Size', fontsize=11)
ax.set_ylabel('Bandwidth (GB/s) — higher is better', fontsize=11)
ax.set_title('Device → Host Transfer', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(transfer_sizes, rotation=30)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Allocation time
ax = axes[2]
alloc_sizes = ['10MB', '100MB', '500MB', '1000MB']
x_pos_a = np.arange(len(alloc_sizes))

for i, fw in enumerate(frameworks):
    times = []
    for als in alloc_sizes:
        val = safe_get(data[fw], 'benchmarks', 'memory', 'allocation', als, 'mean_ms', default=0)
        times.append(val if val else 0)
    mask = [t > 0 for t in times]
    positions = [x_pos_a[j] + offsets[i] for j in range(len(alloc_sizes)) if mask[j]]
    values = [t for t in times if t > 0]
    if values:
        ax.bar(positions, values, bar_width * 0.9, label=fw, color=COLORS.get(fw, 'gray'), alpha=0.85)

ax.set_xlabel('Allocation Size', fontsize=11)
ax.set_ylabel('Time (ms) — lower is better', fontsize=11)
ax.set_title('GPU Memory Allocation', fontsize=13)
ax.set_xticks(x_pos_a)
ax.set_xticklabels(alloc_sizes, rotation=30)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_memory.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: comparison_memory.png")

✅ Saved: comparison_memory.png


In [7]:
# ============================================================
# COMPARISON - Cell 7: Comprehensive Summary Table
# ============================================================

print("=" * 90)
print("COMPREHENSIVE BENCHMARK SUMMARY")
print("=" * 90)

# Build summary table
summary_rows = []

def add_row(category, benchmark, metric, unit):
    row = {'Category': category, 'Benchmark': benchmark, 'Unit': unit}
    for fw in frameworks:
        val = None
        try:
            if isinstance(metric, (list, tuple)):
                val = safe_get(data[fw], *metric, default=None)
            elif callable(metric):
                val = metric(data[fw])
        except:
            val = None
        row[fw] = val
    summary_rows.append(row)

# MatMul
add_row('Compute', 'MatMul 4096 FP32',
        ('benchmarks', 'matmul', 'float32', '4096', 'tflops'), 'TFLOPS↑')
add_row('Compute', 'MatMul 4096 FP16',
        ('benchmarks', 'matmul', 'float16', '4096', 'tflops'), 'TFLOPS↑')
add_row('Compute', 'MatMul 8192 FP32',
        ('benchmarks', 'matmul', 'float32', '8192', 'tflops'), 'TFLOPS↑')

# Element-wise
add_row('Elementwise', 'ReLU 50M',
        ('benchmarks', 'elementwise', '50M', 'relu', 'mean_ms'), 'ms↓')
add_row('Elementwise', 'GELU 50M',
        ('benchmarks', 'elementwise', '50M', 'gelu', 'mean_ms'), 'ms↓')
add_row('Elementwise', 'Exp 50M',
        ('benchmarks', 'elementwise', '50M', 'exp', 'mean_ms'), 'ms↓')

# Reductions
add_row('Reduction', 'Sum 50M',
        ('benchmarks', 'reductions', '50M', 'sum', 'mean_ms'), 'ms↓')
add_row('Reduction', 'Sort 50M',
        ('benchmarks', 'reductions', '50M', 'sort', 'mean_ms'), 'ms↓')

# Conv2D
add_row('Conv2D', 'ResNet-stem Fwd',
        ('benchmarks', 'conv2d', 'ResNet-stem', 'forward', 'mean_ms'), 'ms↓')
add_row('Conv2D', 'VGG-style Fwd+Bwd',
        ('benchmarks', 'conv2d', 'VGG-style', 'fwd_bwd', 'mean_ms'), 'ms↓')

# Attention
add_row('Attention', 'BERT-attn Fwd',
        ('benchmarks', 'layers', 'attention', 'BERT-attn', 'forward', 'mean_ms'), 'ms↓')
add_row('Attention', 'LLM-attn Fwd',
        ('benchmarks', 'layers', 'attention', 'LLM-attn', 'forward', 'mean_ms'), 'ms↓')

# CNN Training
add_row('CNN Train', 'ResNet Infer BS=32',
        ('benchmarks', 'cnn_training', 'batch_32', 'inference', 'throughput_imgs_per_sec'), 'img/s↑')
add_row('CNN Train', 'ResNet Train BS=32',
        ('benchmarks', 'cnn_training', 'batch_32', 'training', 'throughput_imgs_per_sec'), 'img/s↑')
add_row('CNN Train', 'ResNet AMP BS=32',
        ('benchmarks', 'cnn_training', 'amp_batch_32', 'training', 'throughput_imgs_per_sec'), 'img/s↑')

# Transformer Training
add_row('Transformer', 'BERT-base Infer',
        ('benchmarks', 'transformer_training', 'BERT-base-like', 'inference', 'tokens_per_sec'), 'tok/s↑')
add_row('Transformer', 'BERT-base Train',
        ('benchmarks', 'transformer_training', 'BERT-base-like', 'training', 'tokens_per_sec'), 'tok/s↑')

# Memory Transfer
add_row('Memory', 'H2D 100MB',
        ('benchmarks', 'memory', 'h2d', '100MB', 'bandwidth_gbs'), 'GB/s↑')
add_row('Memory', 'D2H 100MB',
        ('benchmarks', 'memory', 'd2h', '100MB', 'bandwidth_gbs'), 'GB/s↑')

# Print table
print(f"\n{'Category':<14} {'Benchmark':<24} {'Unit':<10}", end='')
for fw in frameworks:
    print(f" {fw:>14}", end='')
print(f" {'Winner':>14}")
print("-" * (14 + 24 + 10 + len(frameworks) * 15 + 15))

for row in summary_rows:
    print(f"{row['Category']:<14} {row['Benchmark']:<24} {row['Unit']:<10}", end='')

    values = {}
    for fw in frameworks:
        val = row[fw]
        if val is not None and val != 0:
            values[fw] = val
            print(f" {val:>14.2f}", end='')
        else:
            print(f" {'N/A':>14}", end='')

    # Determine winner
    if values:
        higher_better = '↑' in row['Unit']
        if higher_better:
            winner = max(values, key=values.get)
        else:
            winner = min(values, key=values.get)
        print(f" {'🏆 ' + winner:>14}", end='')
    print()

# --- Wins Summary ---
print("\n" + "=" * 70)
print("WINS SUMMARY")
print("=" * 70)

wins = {fw: 0 for fw in frameworks}
for row in summary_rows:
    values = {}
    for fw in frameworks:
        val = row[fw]
        if val is not None and val != 0:
            values[fw] = val
    if values:
        higher_better = '↑' in row['Unit']
        if higher_better:
            winner = max(values, key=values.get)
        else:
            winner = min(values, key=values.get)
        wins[winner] += 1

total_benchmarks = len(summary_rows)
for fw in frameworks:
    pct = (wins[fw] / total_benchmarks * 100) if total_benchmarks > 0 else 0
    bar = '█' * int(pct / 2)
    print(f"  {fw:<14}: {wins[fw]:>3} / {total_benchmarks} wins ({pct:5.1f}%) {bar}")

print("\n" + "=" * 70)
print("NOTES:")
print("  ↑ = higher is better")
print("  ↓ = lower is better")
print("  Results may vary based on GPU model, driver version, and library version")
print("  All benchmarks use proper GPU synchronization")
print("=" * 70)

COMPREHENSIVE BENCHMARK SUMMARY

Category       Benchmark                Unit              PyTorch     TensorFlow            JAX         Winner
------------------------------------------------------------------------------------------------------------
Compute        MatMul 4096 FP32         TFLOPS↑              7.68           2.24           6.20      🏆 PyTorch
Compute        MatMul 4096 FP16         TFLOPS↑             13.70           4.54          11.61      🏆 PyTorch
Compute        MatMul 8192 FP32         TFLOPS↑              7.69           3.77           5.88      🏆 PyTorch
Elementwise    ReLU 50M                 ms↓                  1.80         111.49           2.39      🏆 PyTorch
Elementwise    GELU 50M                 ms↓                  1.80         111.81           2.38      🏆 PyTorch
Elementwise    Exp 50M                  ms↓                  1.80         107.05           2.37      🏆 PyTorch
Reduction      Sum 50M                  ms↓                  0.83           1.07 

In [8]:
# ============================================================
# COMPARISON - Cell 8: Radar Chart & Overall Score
# ============================================================

# Normalize scores for radar chart (0-1 scale, higher=better)
categories = [
    'MatMul\nFP32', 'MatMul\nFP16', 'Elementwise\nOps', 'Reduction\nOps',
    'Conv2D\nFwd', 'Attention\nFwd', 'CNN\nTraining', 'Transformer\nTraining',
    'Memory\nTransfer', 'Mixed\nPrecision'
]

def get_normalized_score(fw, data_dict):
    """Get performance scores normalized to 0-1 for radar chart"""
    scores = {}

    raw = {}
    # Matmul FP32 (TFLOPS, higher better)
    raw['MatMul\nFP32'] = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'matmul', 'float32', '4096', 'tflops', default=0) or 0
    # Matmul FP16
    raw['MatMul\nFP16'] = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'matmul', 'float16', '4096', 'tflops', default=0) or 0
    # Elementwise (inverse of time, higher better)
    ew_time = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'elementwise', '50M', 'relu', 'mean_ms', default=999) or 999
    raw['Elementwise\nOps'] = 1.0 / max(ew_time, 0.001)
    # Reduction
    red_time = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'reductions', '50M', 'sum', 'mean_ms', default=999) or 999
    raw['Reduction\nOps'] = 1.0 / max(red_time, 0.001)
    # Conv2D
    conv_time = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'conv2d', 'ResNet-stem', 'forward', 'mean_ms', default=999) or 999
    raw['Conv2D\nFwd'] = 1.0 / max(conv_time, 0.001)
    # Attention
    attn_time = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'layers', 'attention', 'BERT-attn', 'forward', 'mean_ms', default=999) or 999
    raw['Attention\nFwd'] = 1.0 / max(attn_time, 0.001)
    # CNN Training throughput
    raw['CNN\nTraining'] = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'cnn_training', 'batch_32', 'training', 'throughput_imgs_per_sec', default=0) or 0
    # Transformer Training
    raw['Transformer\nTraining'] = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'transformer_training', 'BERT-base-like', 'training', 'tokens_per_sec', default=0) or 0
    # Memory transfer
    raw['Memory\nTransfer'] = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'memory', 'h2d', '100MB', 'bandwidth_gbs', default=0) or 0
    # Mixed precision
    raw['Mixed\nPrecision'] = safe_get(data_dict.get(fw, {}),
        'benchmarks', 'cnn_training', 'amp_batch_32', 'training', 'throughput_imgs_per_sec', default=0) or 0

    return raw

# Get raw scores
all_raw = {}
for fw in frameworks:
    all_raw[fw] = get_normalized_score(fw, data)

# Normalize to 0-1 range per category
normalized = {fw: {} for fw in frameworks}
for cat in categories:
    vals = [all_raw[fw].get(cat, 0) for fw in frameworks]
    max_val = max(vals) if max(vals) > 0 else 1
    for fw in frameworks:
        normalized[fw][cat] = all_raw[fw].get(cat, 0) / max_val if max_val > 0 else 0

# Create radar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8),
                                subplot_kw={'projection': 'polar'} if True else {})

# Radar chart
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

fig = plt.figure(figsize=(20, 8))
ax1 = fig.add_subplot(121, projection='polar')

for fw in frameworks:
    values = [normalized[fw].get(cat, 0) for cat in categories]
    values += values[:1]
    ax1.plot(angles, values, 'o-', linewidth=2, label=fw, color=COLORS.get(fw, 'gray'))
    ax1.fill(angles, values, alpha=0.1, color=COLORS.get(fw, 'gray'))

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories, fontsize=9)
ax1.set_ylim(0, 1.1)
ax1.set_title('Normalized Performance Comparison\n(1.0 = best in category)',
              fontsize=14, fontweight='bold', pad=20)
ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

# Overall score bar chart
ax2 = fig.add_subplot(122)
overall_scores = {}
for fw in frameworks:
    scores = [normalized[fw].get(cat, 0) for cat in categories]
    overall_scores[fw] = np.mean(scores)

fw_sorted = sorted(overall_scores.keys(), key=lambda x: overall_scores[x], reverse=True)
bars = ax2.barh(range(len(fw_sorted)), [overall_scores[fw] for fw in fw_sorted],
               color=[COLORS.get(fw, 'gray') for fw in fw_sorted], alpha=0.85, height=0.5)

for i, (fw, bar) in enumerate(zip(fw_sorted, bars)):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{overall_scores[fw]:.3f}', va='center', fontsize=12, fontweight='bold')

ax2.set_yticks(range(len(fw_sorted)))
ax2.set_yticklabels(fw_sorted, fontsize=13)
ax2.set_xlabel('Average Normalized Score — higher is better', fontsize=12)
ax2.set_title('Overall Performance Score', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 1.15)
ax2.grid(axis='x', alpha=0.3)

# Add ranking
for i, fw in enumerate(fw_sorted):
    medal = ['🥇', '🥈', '🥉'][i] if i < 3 else f'#{i+1}'
    ax2.text(-0.05, i, medal, va='center', ha='right', fontsize=16)

plt.tight_layout()
plt.savefig('comparison_overall.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 70)
print("FINAL RANKING")
print("=" * 70)
for i, fw in enumerate(fw_sorted):
    medal = ['🥇', '🥈', '🥉'][i] if i < 3 else f'#{i+1}'
    print(f"  {medal} {fw}: {overall_scores[fw]:.4f} avg normalized score")
print("=" * 70)
print("\n✅ All comparison charts saved!")
print("   📊 comparison_matmul.png")
print("   📊 comparison_elementwise.png")
print("   📊 comparison_layers.png")
print("   📊 comparison_training.png")
print("   📊 comparison_memory.png")
print("   📊 comparison_overall.png")


FINAL RANKING
  🥇 PyTorch: 0.9853 avg normalized score
  🥈 JAX: 0.7006 avg normalized score
  🥉 TensorFlow: 0.3679 avg normalized score

✅ All comparison charts saved!
   📊 comparison_matmul.png
   📊 comparison_elementwise.png
   📊 comparison_layers.png
   📊 comparison_training.png
   📊 comparison_memory.png
   📊 comparison_overall.png
